In [1]:
import pickle as pkl
import numpy as np

train_data  = np.load('mydata/train.npy', allow_pickle=True)

In [2]:
first_sample = train_data[0]

In [3]:
with open('mydata/network_porto/porto_edges_new_simplify.pkl', 'rb') as f:
    edgeinfo = pkl.load(f)
with open('mydata/network_porto/porto_nodes_new.pkl', 'rb') as f:
    nodeinfo = pkl.load(f)

In [4]:
print(edgeinfo[0])
print(nodeinfo['25503936'])

['motorway_link', 32.3884588871153, '25503936', '4722746638']
(-8.6406364, 41.1660713, 3.0)


In [5]:
import torch
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import Dataset

In [10]:
highway_type = set()
for highway, length, u, v in edgeinfo.values():
    highway_type.add(highway)
import ast

pure_types = set()
conjoined_types = set()

for hw in highway_type:
    try:
        parsed = ast.literal_eval(hw)
        if isinstance(parsed, list):
            conjoined_types.add(tuple(parsed))
            for t in parsed:
                pure_types.add(t)
        else:
            pure_types.add(hw)
    except (ValueError, SyntaxError):
        # plain string like 'residential', 'motorway' etc
        pure_types.add(hw)

print("Pure types:", pure_types)
print("Conjoined types:", conjoined_types)

Pure types: {'trunk_link', 'residential', 'unclassified', 'secondary', 'primary_link', 'tertiary', 'crossing', 'tertiary_link', 'motorway_link', 'secondary_link', 'road', 'primary', 'trunk', 'busway', 'living_street', 'motorway'}
Conjoined types: {('secondary', 'primary'), ('tertiary', 'residential'), ('secondary', 'motorway_link'), ('secondary', 'secondary_link'), ('tertiary', 'unclassified'), ('residential', 'living_street'), ('tertiary_link', 'tertiary'), ('motorway', 'motorway_link')}


In [7]:
print(highway_type)

{"['residential', 'living_street']", "['tertiary', 'unclassified']", 'trunk_link', "['tertiary', 'residential']", "['tertiary_link', 'tertiary']", 'primary_link', 'motorway_link', 'residential', "['motorway', 'motorway_link']", 'crossing', 'tertiary_link', 'busway', 'trunk', "['secondary', 'motorway_link']", "['secondary', 'secondary_link']", 'living_street', 'motorway', 'road', 'unclassified', "['secondary', 'primary']", 'tertiary', 'secondary_link', 'primary', 'secondary'}


In [15]:
pure_types = sorted(pure_types)
conjoined_types = sorted(conjoined_types)

# Build lookup dicts for O(1) access instead of .index() which is O(n)
pure_encoding = {hw: i+1 for i, hw in enumerate(pure_types)}        # 1-indexed, 0 = unknown
conjoined_encoding = {hw: i+1 for i, hw in enumerate(conjoined_types)}

def encode_highway(hw):
    try:
        parsed = ast.literal_eval(hw)
        if isinstance(parsed, list):
            return conjoined_encoding.get(tuple(parsed), 0), True
        else:
            return pure_encoding.get(hw, 0), False
    except (ValueError, SyntaxError):
        return pure_encoding.get(hw, 0), False

In [16]:
def collate(data):

    travel_time = np.array([d[-1] for d in data], dtype=np.float32)

    linkids = []
    inds = []
    for _, l in enumerate(data):
        linkids.append(np.asarray(l[1]))
        inds.append(l[0])
    lens = np.asarray([len(k) for k in linkids], dtype=np.int16)
    
    def info(xs):
        infos = []
        lengths = []
        is_boundaries = []
        for x in xs:
            highway, length, u, v = edgeinfo[x]
            enc, is_boundary = encode_highway(highway)
            infos.append(enc)
            lengths.append(length)
            is_boundaries.append(is_boundary)
        return infos, lengths, is_boundaries

    con_links = []
    con_lengths = []
    con_boundaries = []
    for b in linkids:
        hw, ln, bd = info(b)
        con_links.append(hw)
        con_lengths.append(ln)
        con_boundaries.append(bd)
    
    return {
        'links': con_links,           # [[encoded_highway_type]]
        'lengths': con_lengths,        # [[segment_length]]
        'is_boundary': con_boundaries, # [[bool]] — True if conjoined segment
        'lens': lens,
        'inds': inds,
    }, travel_time

In [17]:

class Datadict(Dataset):
    def __init__(self, inputs):
        self.content = inputs

    def __getitem__(self, idx):
        return self.content[idx]

    def __len__(self):
        return len(self.content)

loader = DataLoader(Datadict(train_data), batch_size=16, collate_fn=collate, pin_memory=True, shuffle=False)


In [51]:
dow_model = PositionalEncoding1D(d_model=16)
doy_model = PositionalEncoding1D(d_model=16)
minute_model = PositionalEncoding1D(d_model=16)

In [18]:
from tqdm import tqdm

with torch.no_grad():
    for batch, travel_time in loader:
        print(batch['links'])
        break

[[8, 12, 8, 10, 10, 10, 10, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 8, 8, 3, 8, 8, 8], [6, 10, 8, 8, 8, 8, 8, 8, 8, 8, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 8, 8, 8, 8, 10, 10, 12, 12, 12, 8, 3, 3, 3, 10, 10], [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 1, 7, 12, 12, 12, 12, 12, 12, 12, 10, 10, 8], [10, 10, 10, 10, 10, 10, 10, 10, 12, 12, 12, 12, 12, 12, 12, 12, 12, 8, 8, 8, 12, 10, 10, 10, 10, 10, 12, 12, 12, 12, 8, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8], [6, 6, 6, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 5, 5, 5, 5, 5, 5, 4, 5, 5, 10, 12, 12, 12, 10], [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 5, 11, 10, 10, 10, 10, 10, 10, 10, 8, 8, 8, 8, 8, 8, 8, 10, 10, 10, 8, 8], [10, 10, 10, 10, 11, 10, 10, 8, 12, 12, 5, 4, 4, 4, 5, 5, 5, 5, 4, 5, 5, 10, 10, 10, 10, 11, 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("trip_time_features.npz")

dow = data["day_of_week"]
doy = data["day_of_year"]
minute = data["minute_of_day"]
travel_time = data["travel_time"]
length = data["lengths"]

print(dow.shape, doy.shape, minute.shape, travel_time.shape, length.shape)

In [ ]:
plt.figure(figsize=(6,4))
plt.boxplot(
    [travel_time[dow == i] for i in range(7)],
    labels=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
)
plt.xlabel("Day of Week")
plt.ylabel("Travel Time")
plt.title("Travel Time vs Day of Week")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(minute, travel_time, s=1, alpha=0.2)
plt.xlabel("Minute of Day")
plt.ylabel("Travel Time")
plt.title("Travel Time vs Minute of Day")
plt.tight_layout()
plt.show()

In [ ]:
bins = 144  # 10-minute bins
bin_ids = (minute / (1440 / bins)).astype(int)

mean_tt = np.zeros(bins)
for i in range(bins):
    mean_tt[i] = travel_time[bin_ids == i].mean()

plt.figure(figsize=(8,4))
plt.plot(np.linspace(0, 1440, bins), mean_tt)
plt.xlabel("Minute of Day")
plt.ylabel("Mean Travel Time")
plt.title("Average Daily Congestion Pattern")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(doy, travel_time, s=1, alpha=0.2)
plt.xlabel("Day of Year")
plt.ylabel("Travel Time")
plt.title("Travel Time vs Day of Year")
plt.tight_layout()
plt.show()

In [ ]:
sin_time = np.sin(2 * np.pi * minute / 1440)
cos_time = np.cos(2 * np.pi * minute / 1440)

plt.figure(figsize=(6,4))
plt.scatter(sin_time, travel_time, s=1, alpha=0.2, label="sin")
plt.scatter(cos_time, travel_time, s=1, alpha=0.2, label="cos")
plt.xlabel("Sinusoidal Time Encoding")
plt.ylabel("Travel Time")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
time_bins = [
    ("Night (0–5h)", 0, 300),
    ("Morning (7–9h)", 420, 540),
    ("Midday (11–13h)", 660, 780),
    ("Evening (16–18h)", 960, 1080),
]

In [ ]:
distance_mean = np.mean(length)
distance_std = np.std(length)
q1 = np.percentile(length, 25)
q3 = np.percentile(length, 75)

print(f"Distance Mean: {distance_mean:.2f}")
print(f"Distance Standard Deviation: {distance_std:.2f}")
print(f"Distance 25th Percentile: {q1:.2f}")
print(f"Distance 75th Percentile: {q3:.2f}")

In [ ]:
plt.figure()
plt.hist(length, bins=50, density=True, alpha=0.7)
plt.axvline(distance_mean, linestyle='--', linewidth=2, label='Mean')
plt.axvline(q1, linestyle=':', linewidth=2, label='25th percentile')
plt.axvline(q3, linestyle=':', linewidth=2, label='75th percentile')

plt.xlabel("Trip Distance")
plt.ylabel("Density")
plt.title("Distribution of Trip Distance with Summary Statistics")
plt.legend()
plt.show()